# 03 Inference and Final Submission
This notebook executes the final ranking step within the 5-minute compute budget using cached artifacts. It runs completely offline.

In [ ]:
%pip install pandas numpy sentence-transformers pyarrow fastparquet python-docx

In [10]:
import os
import time
import json
import re
import numpy as np
import pandas as pd
import subprocess
from docx import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

base_dir = "/content/ai-candidate-ranking"
if not os.path.exists(base_dir):
    base_dir = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
    
raw_dir = os.path.join(base_dir, "data", "raw")
processed_dir = os.path.join(base_dir, "data", "processed")
outputs_dir = os.path.join(base_dir, "outputs", "submissions")
os.makedirs(outputs_dir, exist_ok=True)

### 1. Load Processed Artifacts

In [ ]:
print("Loading precomputed artifacts...")
start_time = time.time()

ids_path = os.path.join(processed_dir, "candidate_ids.npy")
candidate_ids = np.load(ids_path, allow_pickle=True)

emb_path = os.path.join(processed_dir, "embeddings.npy")
cand_embeddings = np.load(emb_path)

feat_path = os.path.join(processed_dir, "candidates_feather.parquet")
features_df = pd.read_parquet(feat_path)

print(f"Loaded {len(candidate_ids)} candidates.")
print(f"Loading time: {time.time() - start_time:.2f} seconds")

### 2. Build and Encode JD Text

In [ ]:
def build_jd_text(docx_path):
    try:
        doc = Document(docx_path)
        return "\n".join([p.text.strip() for p in doc.paragraphs if p.text.strip()])
    except Exception as e:
        print(f"Error reading JD: {e}")
        return ""

jd_path = os.path.join(raw_dir, "job_description.docx")
jd_text = build_jd_text(jd_path)

# Load model avoiding network downloads by relying on local cache
model_name = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading {model_name}...")
model = SentenceTransformer(model_name)

print("Encoding JD text...")
jd_embedding = model.encode([jd_text]) if jd_text else np.array([])

### 3. Compute Semantic Similarity

In [ ]:
print("Computing cosine similarities...")
sim_start = time.time()

if cand_embeddings.shape[0] > 0 and jd_embedding.shape[0] > 0:
    similarities = cosine_similarity(jd_embedding, cand_embeddings)[0]
else:
    similarities = np.zeros(len(candidate_ids))

print(f"Similarity computation took: {time.time() - sim_start:.3f} seconds")

sim_df = pd.DataFrame({"candidate_id": candidate_ids, "semantic_similarity": similarities})
full_df = pd.merge(features_df, sim_df, on="candidate_id", how="inner")

# --- Normalize features that are out of [0,1] in the precomputed parquet ---
if full_df["github_fit_score"].max() > 1.0:
    full_df["github_fit_score"] = (full_df["github_fit_score"] / 100.0).clip(0, 1)
    print("  Normalized github_fit_score → [0,1]")
if full_df["availability_score"].max() > 1.0:
    amax = full_df["availability_score"].max()
    full_df["availability_score"] = (full_df["availability_score"] / amax).clip(0, 1)
    print(f"  Normalized availability_score → [0,1] (was 0–{amax:.1f})")
if full_df["ai_skill_depth_score"].max() > 1.0:
    dmax = full_df["ai_skill_depth_score"].max()
    full_df["ai_skill_depth_score"] = (full_df["ai_skill_depth_score"] / dmax).clip(0, 1)
    print(f"  Normalized ai_skill_depth_score → [0,1] (was 0–{dmax:.1f})")

print(f"\nMerged DataFrame: {full_df.shape[0]} rows, {full_df.shape[1]} cols")


### 4. Final Scoring & Reasoning Generation

In [ ]:
# --- Recalibrated Scoring (total positive weights = 0.88) ---
weights = {
    "semantic": 0.30, "role": 0.10, "product": 0.08, "exp_band": 0.10,
    "ml_years": 0.10, "ai_depth": 0.05, "avail": 0.04, "rel": 0.04,
    "github": 0.03, "geo": 0.02, "work_mode": 0.02
}
alpha = 0.15  # Notice period penalty scale
beta  = 0.30  # Honeypot penalty scale

max_ml = full_df["ml_years_estimate"].max() if "ml_years_estimate" in full_df.columns else 1.0
if max_ml == 0: max_ml = 1.0

def compute_final_score(row):
    score = (
        row.get("semantic_similarity", 0) * weights["semantic"] +
        row.get("role_title_score", 0)    * weights["role"] +
        row.get("product_company_score", 0) * weights["product"] +
        row.get("experience_band_match", 0) * weights["exp_band"] +
        (row.get("ml_years_estimate", 0) / max_ml) * weights["ml_years"] +
        min(row.get("ai_skill_depth_score", 0), 1.0) * weights["ai_depth"] +
        row.get("availability_score", 0)  * weights["avail"] +
        row.get("reliability_score", 0)   * weights["rel"] +
        row.get("github_fit_score", 0)    * weights["github"] +
        row.get("geo_fit_score", 0)       * weights["geo"] +
        row.get("work_mode_fit", 0)       * weights["work_mode"]
    )
    score -= row.get("notice_period_penalty", 0) * alpha
    score -= row.get("honeypot_risk_score", 0)   * beta
    return max(0.0, min(1.0, score))

full_df["final_score"] = full_df.apply(compute_final_score, axis=1)

# --- Diagnostics ---
print("=== final_score distribution (all 100k) ===")
print(full_df["final_score"].describe())
print()

# Filter honeypots and sort
filtered_df = full_df[full_df["honeypot_risk_score"] < 0.6].copy()
ranked_df = filtered_df.sort_values(by="final_score", ascending=False).head(100).copy()

print("Extracting raw details for top candidates to generate reasoning...")
top_100_ids = set(ranked_df["candidate_id"].values)
cand_details = {}

cands_path = os.path.join(raw_dir, "candidates.jsonl")

try:
    with open(cands_path, 'r') as f:
        first_char = f.read(1)
    if first_char == '[':
        with open(cands_path, "r") as f:
            data = json.load(f)
            for cand in data:
                cid = cand.get("candidate_id")
                if cid in top_100_ids: cand_details[cid] = cand
    else:
        with open(cands_path, "r") as f:
            for line in f:
                if not line.strip(): continue
                cand = json.loads(line)
                cid = cand.get("candidate_id")
                if cid in top_100_ids: cand_details[cid] = cand
except Exception as e:
    print(f"Could not load raw data: {e}")

# --- JD-aware keyword patterns based on job_description.docx ---
import re
jd_retrieval_kw = re.compile(
    r"search|retrieval|ranking|recommendation|information retrieval|"
    r"relevance|re-?rank|query|candidate.?match",
    re.IGNORECASE,
)
jd_embedding_kw = re.compile(
    r"embedding|vector|faiss|milvus|pgvector|annoy|"
    r"sentence.?transform|cosine|semantic.?similar",
    re.IGNORECASE,
)
jd_ml_kw = re.compile(
    r"machine learning|deep learning|nlp|pytorch|tensorflow|"
    r"mlflow|llm|ai|neural|transformer|fine.?tun",
    re.IGNORECASE,
)
jd_eval_kw = re.compile(
    r"ndcg|map@|precision@|recall@|eval|evaluation|a/b test",
    re.IGNORECASE,
)
jd_infra_kw = re.compile(
    r"mlops|deploy|deployment|pipeline|docker|kubernetes|ci/cd|"
    r"airflow|production|serving|latency",
    re.IGNORECASE,
)

def generate_reasoning(row):
    cid = row["candidate_id"]
    yoe = float(row.get("total_years_experience", 0) or 0)
    ai_count = int(row.get("ai_core_skills_count", 0) or 0)
    sim = float(row.get("semantic_similarity", 0) or 0)
    ml_years = float(row.get("ml_years_estimate", 0) or 0)
    role_score = float(row.get("role_title_score", 0) or 0)
    prod_score = float(row.get("product_company_score", 0) or 0)
    honeypot = float(row.get("honeypot_risk_score", 0) or 0)

    cand = cand_details.get(cid, {})
    profile = cand.get("profile", {})
    title = profile.get("current_title", row.get("current_title", "Professional"))
    industry = profile.get("current_industry", "")
    signals = cand.get("redrob_signals", {})
    skills = cand.get("skills", [])
    career = cand.get("career_history", [])

    # Extract skill names from the actual profile (no hallucination)
    skill_names = [
        s.get("name", str(s)) if isinstance(s, dict) else str(s)
        for s in skills
    ]

    retrieval_skills = [s for s in skill_names if jd_retrieval_kw.search(s)][:2]
    embedding_skills = [s for s in skill_names if jd_embedding_kw.search(s)][:2]
    ml_skills = [s for s in skill_names if jd_ml_kw.search(s)][:3]
    eval_skills = [s for s in skill_names if jd_eval_kw.search(s)][:2]
    infra_skills = [s for s in skill_names if jd_infra_kw.search(s)][:2]

    # Behavioral signals
    try:
        resp_rate = float(signals.get("recruiter_response_rate", 0) or 0)
    except Exception:
        resp_rate = 0.0
    try:
        interview_rate = float(signals.get("interview_completion_rate", 0) or 0)
    except Exception:
        interview_rate = 0.0
    last_active = str(signals.get("last_active_date", "") or "")
    open_to_work = bool(signals.get("open_to_work_flag", False))
    try:
        notice_days = int(signals.get("notice_period_days", 0) or 0)
    except Exception:
        notice_days = 0

    is_services = prod_score < 0.3
    is_product = prod_score >= 0.6

    parts = []

    # 1) Opening: experience tier + ML years
    if yoe >= 7 and sim > 0.45:
        if is_product:
            parts.append(
                f"Seasoned {title} with {yoe:.1f} years of experience and product-company depth"
            )
        else:
            parts.append(f"Senior {title} with {yoe:.1f} years of experience")
        if ml_years >= 2:
            parts[-1] += f", including {ml_years:.1f} years in ML-focused roles."
        else:
            parts[-1] += " and a strong technical foundation."
    elif yoe >= 3:
        parts.append(
            f"Mid-career {title} with {yoe:.1f} years of experience, showing a clear growth trajectory"
        )
        if ml_years > 0:
            parts[-1] += f" and {ml_years:.1f} years of hands-on ML work."
        else:
            parts[-1] += " in software/data engineering."
    else:
        parts.append(f"Early-career {title} with {yoe:.1f} years of experience")
        if ai_count > 0:
            parts[-1] += f", already building competence across {ai_count} AI-relevant skills."
        else:
            parts[-1] += ", with clear potential to grow into the role."

    # 2) JD-theme alignment
    jd_links = []
    if retrieval_skills:
        jd_links.append(f"search/retrieval work ({', '.join(retrieval_skills)})")
    if embedding_skills:
        jd_links.append(f"embedding/vector work ({', '.join(embedding_skills)})")
    if not jd_links and ml_skills:
        jd_links.append(f"ML competencies ({', '.join(ml_skills[:2])})")
    if eval_skills:
        jd_links.append(f"evaluation practice ({', '.join(eval_skills)})")
    if infra_skills:
        jd_links.append(f"ML infrastructure ({', '.join(infra_skills[:2])})")

    if jd_links:
        parts.append(
            "Directly relevant to the JD through "
            + " and ".join(jd_links[:2])
            + "."
        )
    elif ai_count > 0 and ml_skills:
        parts.append(
            f"Brings {ai_count} AI-relevant skills including {', '.join(ml_skills[:2])}, "
            "though search/ranking signals are lighter."
        )
    elif ai_count > 0:
        parts.append(
            f"Lists {ai_count} AI-adjacent skills, but with limited explicit overlap "
            "with the JD’s search/ranking/embedding focus."
        )
    else:
        domain = "IT services" if is_services else (industry.lower() or "a non-ML domain")
        parts.append(
            f"Profile leans toward {domain} with limited direct ML/retrieval signals, "
            "but adjacent technical experience."
        )

    # 3) Behavioral signals
    behaviors = []
    if resp_rate >= 0.7:
        behaviors.append(f"highly responsive to recruiters ({int(resp_rate*100)}% reply rate)")
    if interview_rate >= 0.8:
        behaviors.append(
            f"strong interview follow-through ({int(interview_rate*100)}% completion)"
        )
    if open_to_work:
        behaviors.append("actively open to new roles")
    if last_active and last_active >= "2026-04":
        behaviors.append(f"recently active ({last_active[:10]})")

    if behaviors:
        parts.append("Platform signals: " + ", ".join(behaviors) + ".")
    elif resp_rate > 0 or last_active:
        parts.append(f"Moderate platform engagement (reply rate {int(resp_rate*100)}%).")
    else:
        parts.append("Limited platform activity data available.")

    # 4) Concerns / tradeoffs
    concerns = []
    if is_services and is_product is False:
        concerns.append("background is primarily in IT services rather than product companies")
    if notice_days > 60:
        concerns.append(f"extended notice period ({notice_days} days)")
    if ml_years < 0.5 and ai_count > 3:
        concerns.append(
            f"lists {ai_count} AI skills but only {ml_years:.1f} years of verified ML tenure"
        )
    if honeypot > 0.2:
        concerns.append("some profile inconsistencies flagged")

    if concerns:
        parts.append("Considerations: " + "; ".join(concerns) + ".")

    return " ".join(parts)

ranked_df["reasoning"] = ranked_df.apply(generate_reasoning, axis=1)

print("=== 10 Random Reasoning Samples ===")
for _, r in ranked_df.sample(min(10, len(ranked_df)), random_state=42).iterrows():
    print(f"\nRank {r.get('rank', '?')} | {r['candidate_id']} | score={r.get('final_score', 0):.3f}")
    print(f"  {r['reasoning']}")


### 5. Format Submission

In [ ]:
ranked_df["score"] = ranked_df["final_score"].round(3)
ranked_df["rank"] = range(1, len(ranked_df) + 1)

# Enforce monotonically non-increasing scores
prev_score = ranked_df.iloc[0]["score"]
adjusted_scores = []
for s in ranked_df["score"]:
    if s > prev_score:
        adjusted_scores.append(prev_score)
    else:
        adjusted_scores.append(s)
        prev_score = s
ranked_df["score"] = adjusted_scores

# --- Verify scores are not all identical (saturation check) ---
n_unique = len(set(adjusted_scores))
assert n_unique > 1, (
    f"SCORING BUG: All {len(adjusted_scores)} scores are identical "
    f"({adjusted_scores[0]}). The scoring formula is saturating."
)
print(f"Score diversity: {n_unique} unique values across {len(adjusted_scores)} candidates ✓")
print(f"Score range: {min(adjusted_scores):.4f} – {max(adjusted_scores):.4f}")

submission_df = ranked_df[["candidate_id", "rank", "score", "reasoning"]].copy()

# Spec guardrails
assert len(submission_df) == 100, f"Expected 100 rows, got {len(submission_df)}"

ranks = submission_df["rank"].tolist()
assert sorted(set(ranks)) == list(range(1, 101)), "Ranks must be 1..100 with no gaps or duplicates"

ids = submission_df["candidate_id"].tolist()
assert len(set(ids)) == len(ids), "Duplicate candidate_id values found in submission"

# Check all IDs exist in the original candidates file
import json
import os

cand_ids = set()
cands_path = os.path.join(raw_dir, "candidates.jsonl")
with open(cands_path, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        cand = json.loads(line)
        cid = cand.get("candidate_id")
        if cid:
            cand_ids.add(cid)

missing = [cid for cid in ids if cid not in cand_ids]
assert not missing, f"{len(missing)} candidate_ids not found in candidates.jsonl: {missing[:5]}"

# Score monotonicity and diversity
scores = submission_df["score"].tolist()
mono = all(scores[i] >= scores[i+1] for i in range(len(scores)-1))
assert mono, "Scores must be monotonically non-increasing with rank"

if len(set(scores)) == 1:
    raise AssertionError("All scores are identical – this is discouraged by the spec")

sub_path = os.path.join(outputs_dir, "ranked_candidates.csv")
submission_df.to_csv(sub_path, index=False)
print(f"Saved final submission to {sub_path}")


### 6. Validation and Inspection

In [ ]:
# --- Validation using the OFFICIAL validator from data/raw/ ---
import sys

val_script = os.path.join(raw_dir, "validate_submission.py")
assert os.path.exists(val_script), f"Official validator not found at {val_script}"

meta_path = os.path.join(base_dir, "submission_metadata.yaml")
if not os.path.exists(meta_path):
    meta_path = os.path.join(raw_dir, "submission_metadata_template.yaml")
assert os.path.exists(meta_path), f"Metadata YAML not found at {meta_path}"

print(f"Validator : {val_script}")
print(f"Metadata  : {meta_path}")
print(f"Submission: {sub_path}")

# Verify columns match the spec (candidate_id, rank, score, reasoning — NO job_id)
spec_columns = ["candidate_id", "rank", "score", "reasoning"]
actual_columns = list(submission_df.columns)
assert actual_columns == spec_columns, (
    f"Column mismatch! Expected {spec_columns}, got {actual_columns}"
)
print(f"Columns   : {actual_columns} ✓ (matches submission_spec.docx)")

print("\n=== Running Official Validator ===")
try:
    result = subprocess.run(
        [sys.executable, val_script, sub_path, meta_path],
        capture_output=True, text=True
    )
    stdout = result.stdout.strip()
    stderr = result.stderr.strip()
    print(stdout)
    if stderr:
        print(f"STDERR: {stderr}")

    if result.returncode == 0 and "successful" in stdout.lower():
        print("\n✅ Validation PASSED")
        # Save a validated copy
        validated_path = os.path.join(outputs_dir, "ranked_candidates_validated.csv")
        submission_df.to_csv(validated_path, index=False)
        print(f"Saved validated copy to {validated_path}")
    else:
        # Validator returned non-zero OR output doesn't say "successful"
        err_msg = stderr if stderr else stdout
        print(f"\n❌ Validation FAILED: {err_msg}")
        print("ranked_candidates_validated.csv was NOT overwritten.")

except Exception as e:
    print(f"\n❌ Validation FAILED: {e}")
    print("ranked_candidates_validated.csv was NOT overwritten.")

# --- Inspection ---
print("\n=== Submission Summary ===")
print(f"Rows: {len(submission_df)} | Unique IDs: {submission_df.candidate_id.nunique()}")
print(f"Score range: {submission_df.score.min():.4f} – {submission_df.score.max():.4f}")
print(f"Ranks: {submission_df['rank'].min()} – {submission_df['rank'].max()}")
mono = all(submission_df.score.iloc[i] >= submission_df.score.iloc[i+1] for i in range(len(submission_df)-1))
print(f"Scores monotonically non-increasing: {'✓' if mono else '✗'}")
print(f"Unique reasonings: {submission_df.reasoning.nunique()}/{len(submission_df)}")

print("\n=== Top 10 rows ===")
display(submission_df.head(10))

print("\n=== 5 Random Sample rows ===")
display(submission_df.sample(min(5, len(submission_df))))
